In [ ]:
%load_ext autoreload
%autoreload 2

import functools
print = functools.partial(print, flush=True)

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns

import flexiznam as flz
from cottage_analysis.analysis import common_utils
from cottage_analysis.plotting import depth_selectivity_plots, plotting_utils
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT
from cottage_analysis.plotting import style
from cottage_analysis.summary_analysis import get_session_list
from v1_depth_map.batch_analysis.eye_tracking.analysis import get_data, get_saccades
from v1_depth_map.paths import get_figures_roots
from wayla import eye_io
from wayla.diagnostics import plot_ellipse_on_frame

In [ ]:
# Register the manuscript font faces (Arial regular + bold + italic, Arial Narrow) and
# apply the publication rcParams: vector fonttypes, font sizes, tick/label padding.
# `style.savefig` then expands the SVG `font:` shorthand so Illustrator reads the
# family, size and weight correctly - see cottage_analysis.plotting.style for both.
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT

style.setup_figure_fonts()

In [ ]:
PROJECT = "hey2_3d-vision_foodres_20220101"
flexilims_session = flz.get_flexilims_session(PROJECT)
READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# 1. LOAD RUNNING SPEED / OPTIC FLOW DATA
results_all = pd.read_pickle(READ_ROOT / "supp" / "results_all_rs_supp.pickle")

# Split into 5-depth and 8-depth cohorts
results_all_5depths = results_all[
    (results_all["session"].iloc[:, 0].str.contains("PZAH6.4b"))
    | (results_all["session"].iloc[:, 0].str.contains("PZAG3.4f"))
]
results_all_8depths = results_all[
    ~(
        (results_all["session"].iloc[:, 0].str.contains("PZAH6.4b"))
        | (results_all["session"].iloc[:, 0].str.contains("PZAG3.4f"))
    )
]
print(f"Loaded {len(results_all_5depths)} sessions (5-depth) and {len(results_all_8depths)} sessions (8-depth)")

In [ ]:
# 2. LOAD EYE TRACKING DATA
target_folder = flz.get_data_root(which="processed", project=PROJECT) / PROJECT / "Analysis" / "eye_tracking"
all_data = pd.read_pickle(target_folder / "all_data.pkl")
example_dlc_res = pd.read_pickle(target_folder / "example_dlc_res.pkl")

sess_to_exclude = {
    "PZAH6.4b_S20220516": "Mouse squint too much, cropping issue",
    "PZAH6.4b_S20220429": "Two eye position. Maybe reflection ill detected",
}
all_data = all_data[~all_data.session.isin(sess_to_exclude)]

# Detect saccades per session
saccades_by_sess = {}
filter_window = 5
threshold = 70
for sess_name, gaze_data in all_data.groupby("session"):
    saccades_by_sess[sess_name] = get_saccades(gaze_data, threshold=threshold, filter_window=filter_window)

# Build saccade-by-trial dataframe
sacc_by_trials = []
for sess_name, sess_df in all_data.groupby("session"):
    sacc_df = saccades_by_sess[sess_name]
    trials = sess_df["trial"].dropna().unique()
    for trial in trials:
        trial_df = sess_df[sess_df["trial"] == trial]
        tdict = dict(
            trial=trial,
            session=sess_name,
            depth=int(trial_df.depth.iloc[1] * 100),
            trial_start=trial_df.harptime.iloc[0],
            trial_end=trial_df.harptime.iloc[-1],
        )
        tdict["nsaccades"] = sacc_df[
            (sacc_df["start_time"] >= tdict["trial_start"])
            & (sacc_df["start_time"] <= tdict["trial_end"])
        ].shape[0]
        sacc_by_trials.append(tdict)
sacc_by_trials = pd.DataFrame(sacc_by_trials)
sacc_by_trials["trial_duration"] = sacc_by_trials["trial_end"] - sacc_by_trials["trial_start"]
sacc_by_trials["saccade_rate"] = sacc_by_trials["nsaccades"] / sacc_by_trials["trial_duration"]

dtypes = all_data.dtypes
is_num = dtypes[dtypes == "float64"].index.copy()
is_num = is_num.drop(["trial", "depth"])
bytrialbydepth = all_data.groupby(["session", "depth"])[is_num].aggregate("mean").reset_index()
bytrialbydepth["depth"] = (bytrialbydepth["depth"] * 100).astype(int)
bysess = sacc_by_trials.groupby(["depth", "session"]).aggregate("mean").reset_index()
print("Eye tracking data processed successfully")

In [ ]:
# 3. LOAD EXAMPLE EYE TRACKING FRAME & GAZE TRACE
example_session = "PZAG3.4f_S20220421"
start_frame = 45344
project_recordings = flz.get_entities(datatype="recording", flexilims_session=flexilims_session)
sess_df = flz.get_entity(name=example_session, datatype="session", flexilims_session=flexilims_session)
recording = project_recordings[project_recordings.origin_id == sess_df["id"]]
recording = recording[recording.protocol == "SpheresPermTubeReward"].iloc[0]

camera = flz.Dataset.from_flexilims(name=f"{recording.name}_right_eye_camera", flexilims_session=flexilims_session)
gaze_data, dlc_res, dlc_ds = get_data(
    project=PROJECT,
    mouse=sess_df.genealogy[0],
    session=sess_df.genealogy[-1],
    recording=recording.genealogy[-1],
    filt_window=3,
    verbose=False,
)
eye_params = eye_io.get_eye_parameters(camera, flexilims_session)
dlc_res.columns = dlc_res.columns.droplevel("scorer")
video_file = camera.path_full / camera.extra_attributes["video_file"]
cropping = dlc_ds.extra_attributes["cropping"]

cam_data = cv2.VideoCapture(str(video_file))
cam_data.set(cv2.CAP_PROP_POS_FRAMES, start_frame - 1)
ret, frame = cam_data.read()
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
cam_data.release()
gray = gray[cropping[2] : cropping[3], cropping[0] : cropping[1]]
print("Loaded example frame and gaze data")

In [ ]:
# 4. ASSEMBLE COMPOSITE SUPPLEMENTARY FIGURE (Panels A - L)
fig = plt.figure(figsize=(18 * CM, 20 * CM))
fontsize_dict = FONTSIZE_DICT

# Panel Letters with updated y and x coordinates to completely avoid overlapping y-labels
panel_coords = {
    "A": (0.01, 0.985),
    "B": (0.51, 0.985),
    "C": (0.01, 0.835),
    "D": (0.51, 0.835),
    "E": (0.01, 0.685),
    "F": (0.51, 0.685),
    "G": (0.01, 0.535),
    "H": (0.51, 0.535),
    "I": (0.01, 0.380),
    "J": (0.32, 0.380),
    "K": (0.01, 0.165),
    "L": (0.51, 0.165),
}
for p, (px, py) in panel_coords.items():
    fig.text(px, py, p, fontsize=fontsize_dict["panel"], fontweight="bold", va="top")

# Parameters for PSTH
nbins = 60
blank_length = 3
corridor_length = 6
blank_ratio = blank_length / (blank_length * 2 + corridor_length)
corridor_ratio = corridor_length / (blank_length * 2 + corridor_length)

depth_lists = [np.geomspace(0.06, 6, 5), np.geomspace(0.05, 6.4, 8)]
results_cohorts = [results_all_5depths, results_all_8depths]

# ROW 1: Running Speed PSTH (A, B)
for iplot, (results, depth_list) in enumerate(zip(results_cohorts, depth_lists)):
    psth = np.vstack([j for i in results.rs_psth_closedloop.values for j in i]).reshape(
        len(results), len(depth_list) + 1, nbins
    )
    ax = fig.add_axes([0.09 + iplot * 0.50, 0.89, 0.31, 0.08])
    ylim_val = 100 if iplot == 0 else 70
    depth_selectivity_plots.plot_PSTH(
        trials_df=None,
        depth_list=depth_list,
        psth=psth,
        roi=0,
        is_closed_loop=True,
        use_col="RS",
        corridor_length=6,
        blank_length=3,
        nbins=60,
        frame_rate=15,
        fontsize_dict=fontsize_dict,
        linewidth=1.2,
        legend_on=True,
        legend_loc="upper left",
        legend_bbox_to_anchor=(1.02, 1.05),
        show_ci=True,
        ylim=(0, ylim_val),
    )
    ax.set_ylabel("Running speed (cm/s)", fontsize=fontsize_dict["label"])
    ax.set_xlabel("Corridor position (m)", fontsize=fontsize_dict["label"])
    ax.set_yticks(np.linspace(0, ylim_val, 3))

# ROW 2: Average Running Speed (C, D)
for iplot, (results, depth_list) in enumerate(zip(results_cohorts, depth_lists)):
    ax = fig.add_axes([0.09 + iplot * 0.50, 0.74, 0.31, 0.08])
    depth_selectivity_plots.plot_mean_running_speed_alldepths(
        results,
        depth_list,
        fontsize_dict,
        param="RS",
        ylim=(0, 120),
        linewidth=1.5,
        elinewidth=1.5,
        jitter=0.2,
        scatter_markersize=2.5,
        scatter_alpha=0.4,
        capsize=4,
        capthick=1.5,
    )
    ax.set_ylabel("Average running\nspeed (cm/s)", fontsize=fontsize_dict["label"])
    ax.set_xlabel("Virtual depth (cm)", fontsize=fontsize_dict["label"])
    ax.set_yticks([0, 40, 80, 120])

# ROW 3: Optic Flow Speed PSTH (E, F)
for iplot, (results, depth_list) in enumerate(zip(results_cohorts, depth_lists)):
    psth = np.vstack([j for i in results.rs_psth_closedloop.values for j in i]).reshape(
        len(results), len(depth_list) + 1, nbins
    )
    psth_of = np.degrees(psth / np.hstack([depth_list, 1]).reshape(1, -1, 1))
    ax = fig.add_axes([0.09 + iplot * 0.50, 0.59, 0.31, 0.08])
    depth_selectivity_plots.plot_PSTH(
        trials_df=None,
        depth_list=depth_list,
        psth=psth_of[
            :,
            :,
            int(psth_of.shape[2] * blank_ratio - 1) : int(
                psth_of.shape[2] * (blank_ratio + corridor_ratio) + 1
            ),
        ],
        roi=0,
        is_closed_loop=True,
        use_col="OF",
        corridor_length=6,
        blank_length=(corridor_length + blank_length * 2) / psth_of.shape[2] / 2,
        nbins=int(psth_of.shape[2] * corridor_ratio) + 2,
        frame_rate=15,
        fontsize_dict=fontsize_dict,
        linewidth=1.2,
        legend_on=True,
        legend_loc="upper left",
        legend_bbox_to_anchor=(1.02, 1.05),
        show_ci=True,
        ylim=(1e0, 1e3),
    )
    ax.set_yscale("log")
    ax.set_ylabel("Optic flow speed\n(degrees/s)", fontsize=fontsize_dict["label"])
    ax.set_xlabel("Corridor position (m)", fontsize=fontsize_dict["label"])
    ax.set_yticks(np.geomspace(1e0, 1e3, 4))

# ROW 4: Average Optic Flow Speed (G, H)
for iplot, (results, depth_list) in enumerate(zip(results_cohorts, depth_lists)):
    ax = fig.add_axes([0.09 + iplot * 0.50, 0.44, 0.31, 0.08])
    depth_selectivity_plots.plot_mean_running_speed_alldepths(
        results,
        depth_list,
        fontsize_dict,
        param="OF",
        of_threshold=0.01,
        ylim=(5e-1, 1e3),
        linewidth=1.5,
        elinewidth=1.5,
        jitter=0.2,
        scatter_markersize=2.5,
        scatter_alpha=0.4,
        capsize=4,
        capthick=1.5,
    )
    ax.set_ylabel("Average optic flow\nspeed (degrees/s)", fontsize=fontsize_dict["label"])
    ax.set_xlabel("Virtual depth (cm)", fontsize=fontsize_dict["label"])
    ax.set_yticks(np.geomspace(1e0, 1e3, 4))

# ROW 5: Eye Image (I) and Gaze Traces (J)
# Full unclipped image with extended axis height
ax_eye = fig.add_axes([0.08, 0.185, 0.136, 0.19])
ax_eye.imshow(gray, cmap="gray", vmin=0, vmax=150)
eye_pos = dlc_res.iloc[start_frame][[f"eye_{i}" for i in range(1, 13)]]
ref = gaze_data.iloc[start_frame][["reflection_x", "reflection_y"]].values
eye_centre = eye_params["eye_centre"] + ref
pupil_center = gaze_data.iloc[start_frame][["centre_x", "centre_y"]].values
ax_eye.plot([eye_centre[0], pupil_center[0]], [eye_centre[1], pupil_center[1]], c="dodgerblue", lw=1.2)
ax_eye.scatter(*eye_centre, s=7, c="k", zorder=5)
ax_eye.scatter(eye_pos.xs("x", level=1), eye_pos.xs("y", level=1), s=2.5, c="indianred", zorder=11)
plot_ellipse_on_frame(
    ax_eye,
    start_frame,
    gaze_data,
    origin="uncropped",
    dlc_res=dlc_res,
    reflection_fit=None,
    color="dodgerblue",
    alpha=1,
    lw=1.0,
    zorder=12,
)
ax_eye.set_xlim(0, gray.shape[1])
ax_eye.set_ylim(gray.shape[0], 0)
ax_eye.axis("off")

# Panel J: Gaze Traces
ax_trace = fig.add_axes([0.35, 0.21, 0.46, 0.13])
b = start_frame - 1000
e = start_frame + 1000
gd = gaze_data.iloc[b:e]
t0 = gaze_data.harptime.iloc[b]
med_pos = np.nanmedian(gaze_data[["azimuth_filt", "elevation_filt"]].values, axis=0)

ax_trace.plot(gd.harptime.values - t0, gd.elevation_filt.values - np.array(med_pos)[1], c="grey", label="Elevation", lw=1.2, zorder=9)
ax_trace.plot(gd.harptime.values - t0, gd.azimuth_filt.values - np.array(med_pos)[0], c="k", label="Azimuth", lw=1.2, zorder=10)
depths = gd.depth.unique()
depths = sorted(np.round(depths[~np.isnan(depths)] * 100).astype(int))
depth_list_5 = np.sort(bytrialbydepth.depth.unique()).astype("float")

for t, tdf in gd.groupby("trial"):
    ax_trace.axvline(tdf.harptime.iloc[0] - t0, c="gray", lw=0.5)
    ax_trace.axvline(tdf.harptime.iloc[-1] - t0, c="gray", lw=0.5)
    d = np.round(tdf.depth.iloc[0] * 100).astype(int)
    depth_index = list(depths).index(d)
    ax_trace.axvspan(
        tdf.harptime.iloc[0] - t0,
        tdf.harptime.iloc[-1] - t0,
        alpha=0.5,
        color=plotting_utils.get_color(
            value=depth_list_5[depth_index],
            value_min=np.min(depth_list_5),
            value_max=np.max(depth_list_5),
            cmap=cm.cool.reversed(),
            log=True,
        ),
    )
ax_trace.set_ylabel("Eye position (degrees)", fontsize=fontsize_dict["label"])
ax_trace.set_xlabel("Time (s)", fontsize=fontsize_dict["label"])
ax_trace.tick_params(axis="both", which="major", labelsize=fontsize_dict["tick"])
ax_trace.set_xlim(0, 120)
ax_trace.set_ylim(-10, 10)
ax_trace.set_yticks([-10, -5, 0, 5, 10])

# Add custom color legend for depth shades on panel J
for idepth, d in enumerate(depth_list_5):
    c_val = plotting_utils.get_color(
        value=d,
        value_min=np.min(depth_list_5),
        value_max=np.max(depth_list_5),
        cmap=cm.cool.reversed(),
        log=True,
    )
    ax_trace.plot([], [], color=c_val, lw=5, alpha=0.5, label=f"{int(d)} cm")
handles, labels = ax_trace.get_legend_handles_labels()
leg1 = ax_trace.legend(handles[:2], labels[:2], loc="upper right", fontsize=fontsize_dict["legend"], ncol=2, bbox_to_anchor=(1.0, 1.25), columnspacing=0.6, frameon=False)
ax_trace.add_artist(leg1)
ax_trace.legend(handles[2:], labels[2:], loc="upper left", fontsize=fontsize_dict["legend"], bbox_to_anchor=(1.02, 1.05), frameon=False, handlelength=0.8)
sns.despine(ax=ax_trace)

# ROW 6: Eye Movement Velocity (K) and Saccade Rate (L)
eye_summaries = [
    (bytrialbydepth, "velocity", "Eye movement\nspeed (degrees/s)", (0, 20), [0, 5, 10, 15, 20]),
    (bysess, "saccade_rate", "Saccade rate (Hz)", (0, 0.2), [0, 0.05, 0.10, 0.15, 0.20]),
]

for icol, (data, col, ylabel, ylim, yticks) in enumerate(eye_summaries):
    ax = fig.add_axes([0.09 + icol * 0.50, 0.05, 0.31, 0.10])
    depth_list = np.sort(bytrialbydepth.depth.unique()).astype("float")
    for idepth, depth in enumerate(depth_list):
        color = plotting_utils.get_color(
            value=depth_list[idepth],
            value_min=np.min(depth_list),
            value_max=np.max(depth_list),
            cmap=cm.cool.reversed(),
            log=True,
        )
        velocity = data[data["depth"] == depth][col].values
        CI_low, CI_high = common_utils.get_bootstrap_ci(velocity.T, sig_level=0.05)
        mean_velocity = np.nanmean(velocity)

        sns.stripplot(
            x=np.ones(velocity.shape) * idepth,
            y=velocity,
            jitter=0.2,
            edgecolor="white",
            color=color,
            alpha=0.4,
            size=3,
            ax=ax,
        )
        ax.plot(
            [idepth - 0.3, idepth + 0.3],
            [mean_velocity, mean_velocity],
            linewidth=1.5,
            color=color,
            label=f"{int(depth)} cm" if icol == 0 or icol == 1 else None,
        )
        ax.errorbar(
            x=idepth,
            y=mean_velocity,
            yerr=np.array([mean_velocity - CI_low, CI_high - mean_velocity]).reshape(2, 1),
            capsize=4,
            elinewidth=1.5,
            ecolor=color,
            capthick=1.5,
        )
    ax.set_ylabel(ylabel, fontsize=fontsize_dict["label"])
    ax.set_xlabel("Virtual depth (cm)", fontsize=fontsize_dict["label"])
    ax.set_xticks(np.arange(len(depth_list)))
    ax.set_xticklabels((depth_list).astype("int"), fontsize=fontsize_dict["tick"])
    ax.tick_params(axis="both", which="major", labelsize=fontsize_dict["tick"])
    ax.set_ylim(ylim)
    ax.set_yticks(yticks)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=fontsize_dict["legend"], frameon=False, handlelength=0.8)
    sns.despine(ax=ax)

# Save figure
style.savefig(SAVE_ROOT / "figsupp_speed.svg", bbox_inches="tight", dpi=300, fig=fig)
print(f"Saved {SAVE_ROOT}/figsupp_speed.svg")